# BTC Daily Strategy — Research Scratchpad

Exploration only. All reusable logic lives in `src/`; do not hide important
logic in this notebook. This cell-by-cell flow mirrors the harness pipeline.

Run from the project root so `import src...` resolves.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # project root

from src.config import load_config, BacktestConfig
from src import data as data_mod, features as features_mod
from src.strategies import get_strategy
from src.backtest import run_backtest
from src import metrics as metrics_mod

config = load_config('../config.yaml')
bt = BacktestConfig.from_config(config)
bt

In [ ]:
# Use synthetic data offline, or data_mod.load_btc_data(config) for real data.
df = data_mod.generate_synthetic_btc(n_days=1000)
feat = features_mod.build_features(df)
feat.tail()

In [ ]:
raw = get_strategy('sma_long_short')(feat, {'sma_slow': 200})
res = run_backtest(
    feat, raw,
    initial_capital=bt.initial_capital, fee_rate=bt.fee_rate,
    slippage_rate=bt.slippage_rate, execution_lag_days=bt.execution_lag_days,
    min_weight=bt.min_weight, max_weight=bt.max_weight, max_leverage=bt.max_leverage,
    rebalance_policy=bt.rebalance_policy, leverage_breach_action=bt.leverage_breach_action,
    funding_config=bt.funding_config,
)
metrics_mod.compute_metrics(res, bt.initial_capital)

In [ ]:
res['equity_end'].plot(logy=True, figsize=(11, 4), title='Equity curve');